# 01 · Ingestión y transformación del dataset (EA1)

Flujo: **Hugging Face → `HFDatasetLoader` → DataFrame → transformaciones → `DatasetSummarizer` → `reports/data-summary.txt`**.

Dataset: [`Nicolybgs/healthcare_data`](https://huggingface.co/datasets/Nicolybgs/healthcare_data). Ejecuta las celdas en orden.

## 1. Preparar el entorno e importar las clases

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "src" / "config.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))  # permite `import src...` sin rutas absolutas

import pandas as pd
pd.set_option("display.max_columns", 30)
from src import config
from src.data.huggingface_loader import HFDatasetLoader
from src.data import preprocessing as prep
from src.utils.reporter import DatasetSummarizer, save_json
from src.data.preprocessing import PREPROCESSING_DESCRIPTION

D:\Documents\Study\IUDigital\ML\2\healthcare-admission-prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Verificar configuración y splits reales, y cargar el dataset

`available_splits()` consulta Hugging Face, así que no se asume nada. Como el dataset completo (500 000 filas) generaría CSV de ~55 MB, se toma una **muestra aleatoria reproducible** (`seed=42`) para respetar los límites de GitHub; el tamaño se controla en `src/config.py` (`MAX_ROWS`).

In [2]:
loader = HFDatasetLoader(
    config.DATASET_ID, config=config.DATASET_CONFIG, split=config.DATASET_SPLIT,
    max_rows=config.MAX_ROWS, seed=config.RANDOM_SEED,
)
print("Configuraciones y splits disponibles:", loader.available_splits())
raw = loader.load()
loader.info()

Configuraciones y splits disponibles: {'default': ['train']}


{'dataset_id': 'Nicolybgs/healthcare_data',
 'config': 'default',
 'split': 'train',
 'filas_originales_en_hf': 500000,
 'filas_cargadas': 100000,
 'columnas': ['Available Extra Rooms in Hospital',
  'Department',
  'Ward_Facility_Code',
  'doctor_name',
  'staff_available',
  'patientid',
  'Age',
  'gender',
  'Type of Admission',
  'Severity of Illness',
  'health_conditions',
  'Visitors with Patient',
  'Insurance',
  'Admission_Deposit',
  'Stay (in days)'],
 'muestreo': 'muestra aleatoria de 100000 filas (seed=42)'}

## 3. Estructura del DataFrame original: tamaño, tipos y muestra

In [3]:
print(f"Tamaño: {raw.shape[0]} filas x {raw.shape[1]} columnas")
display(raw.dtypes.rename("tipo").to_frame())
display(raw.isna().sum().rename("nulos").to_frame().query("nulos > 0"))
raw.head()

Tamaño: 100000 filas x 15 columnas


,tipo
Available Extra Rooms in Hospital,int64
Department,object
Ward_Facility_Code,object
doctor_name,object
staff_available,int64
patientid,int64
Age,object
gender,object
Type of Admission,object
Severity of Illness,object


,nulos
health_conditions,30262


,Available Extra Rooms in Hospital,Department,Ward_Facility_Code,doctor_name,staff_available,patientid,Age,gender,Type of Admission,Severity of Illness,health_conditions,Visitors with Patient,Insurance,Admission_Deposit,Stay (in days)
0,3,gynecology,B,Dr Olivia,9,85727,31-40,Female,Trauma,Moderate,Other,2,Yes,5023.660629,8
1,5,gynecology,F,Dr Olivia,3,50606,31-40,Female,Trauma,Moderate,None,4,Yes,3997.771046,10
2,4,radiotherapy,E,Dr Sam,1,88248,51-60,Male,Trauma,Moderate,Other,2,Yes,8279.540373,20
3,4,TB & Chest disease,C,Dr Mark,3,49051,21-30,Male,Urgent,Extreme,None,4,No,2446.615280,28
4,20,gynecology,D,Dr Nathan,1,44437,41-50,Female,Trauma,Minor,Other,3,No,3863.117356,11


## 4. Guardar los datos sin transformar (`data/raw`)

In [4]:
config.DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)
raw.to_csv(config.RAW_CSV, index=False)
print("Guardado:", config.RAW_CSV.relative_to(ROOT), f"({config.RAW_CSV.stat().st_size/1e6:.1f} MB)")

Guardado: data\raw\healthcare_raw.csv (9.0 MB)


## 5. Transformaciones

Cada transformación devuelve un DataFrame nuevo y se muestra su efecto. El orden coincide con `preprocess()`.

### 5.1 Tratar nulos y duplicados de forma explícita
* `health_conditions` tiene nulos que en este dataset significan **"sin condición preexistente"**; se rellenan con la categoría `"None"` en lugar de borrar ~30 % de las filas.
* Se eliminan filas duplicadas **exactas** (antes de quitar `patientid`, para no tratar como repetidos a pacientes distintos con características iguales).

In [5]:
df = prep.strip_text_columns(raw)   # limpieza menor: espacios sobrantes
dup_antes = int(df.duplicated().sum())
df1 = prep.drop_duplicates(df)
nulos_antes = int(df1["health_conditions"].isna().sum())
df1 = prep.fill_missing_conditions(df1)
print(f"Duplicados eliminados: {dup_antes} | filas: {len(df)} -> {len(df1)}")
print(f"Nulos en health_conditions: {nulos_antes} -> {int(df1['health_conditions'].isna().sum())}")
df1["health_conditions"].value_counts().to_frame()

Duplicados eliminados: 0 | filas: 100000 -> 100000
Nulos en health_conditions: 30262 -> 0


,count
health_conditions,
None,30262
Other,18950
High Blood Pressure,15738
Diabetes,14762
Asthama,13169
Heart disease,7119


### 5.2 Eliminar `patientid`
Es un identificador único: no tiene valor predictivo y podría inducir al modelo a memorizar pacientes. Se elimina de la versión analítica.

In [6]:
df2 = prep.drop_identifier(df1)
print("patientid" in df2.columns, "|", df1.shape, "->", df2.shape)

False | (100000, 15) -> (100000, 14)


### 5.3 Convertir `Age` de rangos de texto a número
`Age` viene como rangos (`"41-50"`). Los modelos lineales necesitan un valor numérico: se usa el **punto medio** del rango (`45.5`) en la nueva columna `age_midpoint`, que reemplaza a `Age`.

In [7]:
df3 = prep.encode_age_midpoint(df2)
display(df2["Age"].value_counts().sort_index().to_frame())
df3[["age_midpoint"]].drop_duplicates().sort_values("age_midpoint").T

,count
Age,
0-10,684
11-20,9275
21-30,31820
31-40,26821
41-50,16077
51-60,4410
61-70,5265
71-80,3793
81-90,1594


,64,9,3,0,4,2,15,62,22,544
age_midpoint,5.0,15.5,25.5,35.5,45.5,55.5,65.5,75.5,85.5,95.5


### 5.4 Normalizar nombres de columnas a `snake_case`
Nombres como `Stay (in days)` o `Available Extra Rooms in Hospital` son incómodos de usar en código. Se convierten a minúsculas con guiones bajos (`stay_in_days`).

In [8]:
processed = prep.normalize_column_names(df3)
pd.DataFrame({"antes": list(df3.columns), "después": list(processed.columns)})

,antes,después
0,Available Extra Rooms in Hospital,available_extra_rooms_in_hospital
1,Department,department
2,Ward_Facility_Code,ward_facility_code
3,doctor_name,doctor_name
4,staff_available,staff_available
5,gender,gender
6,Type of Admission,type_of_admission
7,Severity of Illness,severity_of_illness
8,health_conditions,health_conditions
9,Visitors with Patient,visitors_with_patient


### 5.5 Comprobación y guardado (`data/processed`)
La función `preprocess()` aplica todos los pasos en el mismo orden; se verifica que da el mismo resultado.

In [9]:
assert processed.equals(prep.preprocess(raw)), "preprocess() debe coincidir con los pasos manuales"
config.DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
processed.to_csv(config.PROCESSED_CSV, index=False)
print("Guardado:", config.PROCESSED_CSV.relative_to(ROOT), f"({config.PROCESSED_CSV.stat().st_size/1e6:.1f} MB)")

Guardado: data\processed\healthcare_processed.csv (8.5 MB)


## 6. Resumen del DataFrame **transformado** con `DatasetSummarizer`

In [10]:
ruta = DatasetSummarizer().save(processed, config.DATA_SUMMARY_TXT)
info = loader.info()
save_json({
    "dataset_id": config.DATASET_ID, "dataset_url": config.DATASET_URL,
    "config": info["config"], "split": info["split"],
    "filas_originales_en_hf": info["filas_originales_en_hf"], "filas_cargadas": info["filas_cargadas"],
    "muestreo": info["muestreo"], "columnas_iniciales": info["columnas"],
    "filas_procesadas": len(processed), "columnas_procesadas": list(processed.columns),
    "preprocesamiento": PREPROCESSING_DESCRIPTION,
}, config.INGESTION_METADATA_JSON)
print("Resumen generado en:", ruta.relative_to(ROOT))
print(ruta.read_text(encoding="utf-8"))

Resumen generado en: reports\data-summary.txt
RESUMEN DEL DATAFRAME
Filas: 100000
Columnas: 14
Filas duplicadas: 0

Nombres de columnas:
available_extra_rooms_in_hospital, department, ward_facility_code, doctor_name, staff_available, gender, type_of_admission, severity_of_illness, health_conditions, visitors_with_patient, insurance, admission_deposit, stay_in_days, age_midpoint

Tipos de datos:
- available_extra_rooms_in_hospital: int64
- department: object
- ward_facility_code: object
- doctor_name: object
- staff_available: int64
- gender: object
- type_of_admission: object
- severity_of_illness: object
- health_conditions: object
- visitors_with_patient: int64
- insurance: object
- admission_deposit: float64
- stay_in_days: int64
- age_midpoint: float64

Valores nulos por columna:
- available_extra_rooms_in_hospital: 0
- department: 0
- ward_facility_code: 0
- doctor_name: 0
- staff_available: 0
- gender: 0
- type_of_admission: 0
- severity_of_illness: 0
- health_conditions: 0
- vis

## 7. Conclusión
Se aplicaron cinco transformaciones (eliminar identificador, `Age` → numérica, `snake_case`, relleno explícito de nulos y eliminación de duplicados). El `.txt` describe el DataFrame **transformado**; los CSV `raw` y `processed` quedan en `data/`.